In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import emoji
import nltk
import contractions  # for expanding English contractions
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping



# Data preparation and preprocessing

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


In [ ]:
class AntonymReplacer:
    def replace(self, word, pos=None):
        antonyms = set()
        for syn in wordnet.synsets(word, pos=pos):
            for lemma in syn.lemmas():
                for antonym in lemma.antonyms():
                    antonyms.add(antonym.name())
        if len(antonyms) == 1:
            return antonyms.pop()
        return None

    def replace_negations(self, tokens):
        i, l = 0, len(tokens)
        words = []
        while i < l:
            word = tokens[i]
            if word == 'not' and i + 1 < l:
                ant = self.replace(tokens[i+1])
                if ant:
                    words.append(ant)
                    i += 2
                    continue
            words.append(word)
            i += 1
        return words

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
replacer = AntonymReplacer()
cleanup_regex = re.compile(r"http\S+|www\S+|@\w+|[^A-Za-z\s]")


In [ ]:
df = pd.read_csv('IMDB_Dataset.csv')
df.head(10)

In [ ]:
def preprocess(sentence):
    # Remove HTML tags
    text = re.sub(r"<.*?>", "", sentence)
    # Expand contractions (e.g., don't -> do not)
    text = contractions.fix(text)
    # Remove numbers
    text = re.sub(r"\d+", "", text)
    # Convert to lowercase
    text = text.lower()
    # Convert emoji to text
    text = emoji.demojize(text)
    # Clean text using precompiled regex
    text = cleanup_regex.sub(' ', text)
    # Tokenize to words (handles whitespace)
    tokens = word_tokenize(text)
    # Remove empty tokens and non-alphabetical tokens
    tokens = [t for t in tokens if t and t.isalpha()]
    # Replace negations using antonyms
    tokens = replacer.replace_negations(tokens)
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

In [ ]:
df['clean_review'] = df['review'].astype(str).apply(preprocess)
df['clean_review'].head(10)

In [ ]:
# Save the cleaned DataFrame to a new CSV file as backup
df.to_csv('cleaned_IMDB_Dataset.csv', index=False)

In [ ]:
df['label'] = df['sentiment'].map({'positive':1, 'negative':0})


In [ ]:
df.info()

# Check for missing values
df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt

count_Class=pd.value_counts(df["sentiment"], sort= True)
count_Class.plot(kind = 'bar',color = ["green","red"])
plt.title('Bar Plot')
plt.show()

In [ ]:
RS = 45

In [ ]:
# Split the dataset into training and testing sets

X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=RS, 
    stratify=y    # ensures train/test have the same positive/negative ratio
)



In [ ]:
# Absolute counts
print("Train set class counts:")
print(y_train.value_counts())
print("\nTest set class counts:")
print(y_test.value_counts())

# Proportions
print("\nTrain set class proportions:")
print(y_train.value_counts(normalize=True))
print("\nTest set class proportions:")
print(y_test.value_counts(normalize=True))


# SVM

### CountVectorizer as feature extraction method

In [ ]:
# CountVectorizer as feature extraction method

from sklearn.svm import LinearSVC

cv = CountVectorizer(min_df=2, max_df=0.9)
X_cv = cv.fit_transform(X_train)

svm_cv = LinearSVC(random_state=RS, dual=False)
# Cross-validation
scores_cv_acc = cross_val_score(svm_cv, X_cv, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print("SVM + CountVectorizer CV Accuracy:", scores_cv_acc)

# Train & evaluate on test set
svm_cv.fit(X_cv, y_train.values.ravel())
X_test_cv = cv.transform(X_test)
y_pred_cv = svm_cv.predict(X_test_cv)
print(classification_report(y_test, y_pred_cv, target_names=['negative','positive']))
pickle.dump(svm_cv, open('svm_countvec_model.pkl','wb'))
pickle.dump(cv, open('count_vectorizer.pkl','wb'))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
cm = confusion_matrix(y_test, y_pred_cv)
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.set_title('Confusion Matrix: SVM + CountVectorizer')
ax.set_xticks([0, 1])
ax.set_xticklabels(['neg', 'pos'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['neg', 'pos'])
plt.ylabel('True label')
plt.xlabel('Predicted label')

# Add the counts to each cell of the confusion matrix
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j],
                horizontalalignment="center",
                color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.show()

### TfidfVectorizer as feature extraction method

In [ ]:
# TfidfVectorizer as feature extraction method

tv = TfidfVectorizer(min_df=2, max_df=0.9, sublinear_tf=True, use_idf=True)
X_tv = tv.fit_transform(X_train)

svm_tv = LinearSVC(random_state=RS, dual=False)
# Cross-validation
scores_tv_acc = cross_val_score(svm_tv, X_tv, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print("SVM + TfidfVectorizer CV Accuracy:", scores_tv_acc)

# Train & evaluate on test set
svm_tv.fit(X_tv, y_train.values.ravel())
X_test_tv = tv.transform(X_test)
y_pred_tv = svm_tv.predict(X_test_tv)
print(classification_report(y_test, y_pred_tv, target_names=['negative','positive']))
pickle.dump(svm_tv, open('svm_tfidf_model.pkl','wb'))
pickle.dump(tv, open('tfidf_vectorizer.pkl','wb'))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
cm = confusion_matrix(y_test, y_pred_tv)
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.set_title('Confusion Matrix: SVM + TfidfVectorizer')
ax.set_xticks([0, 1])
ax.set_xticklabels(['neg', 'pos'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['neg', 'pos'])
plt.ylabel('True label')
plt.xlabel('Predicted label')

# Add the counts to each cell of the confusion matrix
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j],
                horizontalalignment="center",
                color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.show()

# BiLSTM

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)
seq_train = tokenizer.texts_to_sequences(X_train)
seq_test = tokenizer.texts_to_sequences(X_test)
maxlen = 300
X_train_seq = pad_sequences(seq_train, maxlen=maxlen)
X_test_seq  = pad_sequences(seq_test,  maxlen=maxlen)
vocab_size  = len(tokenizer.word_index)+1
print(f"Total vocab size: {vocab_size}")


In [ ]:
def make_embedding_matrix(w2v_model):
    emb = np.zeros((vocab_size,300))
    for w,i in tokenizer.word_index.items():
        if i<vocab_size and w in w2v_model.wv:
            emb[i] = w2v_model.wv[w]
    return emb

In [ ]:

sentences = [t.split() for t in X_train]
w2v_cbow = Word2Vec(
    sentences, vector_size=300, window=7,
    min_count=10, sg=0, workers=8
)
w2v_sg   = Word2Vec(
    sentences, vector_size=300, window=7,
    min_count=10, sg=1, workers=8
)
emb_cbow = make_embedding_matrix(w2v_cbow)
emb_sg   = make_embedding_matrix(w2v_sg)

In [ ]:
def build_bilstm(emb_matrix):
    m = Sequential()
    m.add(Embedding(vocab_size,300,weights=[emb_matrix],input_length=maxlen,trainable=False))
    m.add(Bidirectional(LSTM(128)))
    m.add(Dropout(0.5))
    m.add(Dense(1,activation='sigmoid'))
    m.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
    return m

### BiLSTM - CBOW

In [ ]:
# K-Fold CV for BiLSTM CBOW
kf = KFold(n_splits=5, shuffle=True, random_state=RS)
cbow_acc = []
es = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)

for tr,vl in kf.split(X_train_seq):
    m = build_bilstm(emb_cbow)
    m.fit(
        X_train_seq[tr], y_train.values.ravel()[tr],
        epochs=10, batch_size=128,
        validation_data=(X_train_seq[vl], y_train.values.ravel()[vl]),
        callbacks=[es], verbose=1,
        use_multiprocessing=True, workers=4
    )
    p = (m.predict(X_train_seq[vl])>0.5).astype(int)
    cbow_acc.append(accuracy_score(y_train.values.ravel()[vl], p))
print('BiLSTM+CBOW CV Accuracies:', cbow_acc)
print('Mean:', np.mean(cbow_acc))

In [ ]:
# Final train on full data & test eval for CBOW
es = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)


model_cbow = build_bilstm(emb_cbow)
history_cbow = model_cbow.fit(
    X_train_seq, y_train.values.ravel(),
    epochs=30, batch_size=128,
    validation_split=0.1, callbacks=[es], verbose=1,
    use_multiprocessing=True, workers=4
)
acc_cbow = model_cbow.evaluate(X_test_seq, y_test)[1]
print(f"Test Accuracy CBOW: {acc_cbow}")

In [ ]:
# Visualization: CM for CBOW
yp_cbow = (model_cbow.predict(X_test_seq)>0.5).astype(int)
fig,ax = plt.subplots(1,1,figsize=(5,5))
cm = confusion_matrix(y_test, yp_cbow)
ax.imshow(cm, cmap='Purples')
ax.set_title('BiLSTM+CBOW')
ax.set_xticks([0,1]); ax.set_xticklabels(['neg','pos'])
ax.set_yticks([0,1]); ax.set_yticklabels(['neg','pos'])
plt.xlabel('Pred')
plt.ylabel('True')

# Add text annotations to the plot
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        text_color = "white" if cm[i, j] > cm.max() / 2 else "black" 
        ax.text(j, i, format(cm[i, j], 'd'),  
                ha="center", va="center",
                color=text_color)

plt.show()

In [ ]:
from sklearn.metrics import classification_report

# CBOW BiLSTM classification report
y_pred_cbow = (model_cbow.predict(X_test_seq) > 0.5).astype(int)
print("Classification Report: BiLSTM + CBOW")
print(classification_report(
    y_test,
    y_pred_cbow,
    target_names=['negative','positive']
))

In [ ]:
# Plot CBOW metrics
plt.figure(figsize=(12,4))
# Accuracy
plt.subplot(1,2,1)
plt.plot(history_cbow.history['accuracy'], label='train_acc')
plt.plot(history_cbow.history['val_accuracy'], label='val_acc')
plt.title('CBOW BiLSTM Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend()
# Loss
plt.subplot(1,2,2)
plt.plot(history_cbow.history['loss'], label='train_loss')
plt.plot(history_cbow.history['val_loss'], label='val_loss')
plt.title('CBOW BiLSTM Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend()
plt.show()

### BiLSTM - Skip Gram

In [ ]:
# K-Fold CV for BiLSTM Skip-Gram
sg_acc = []
for tr,vl in kf.split(X_train_seq):
    m = build_bilstm(emb_sg)
    m.fit(
        X_train_seq[tr], y_train.values.ravel()[tr],
        epochs=10, batch_size=128,
        validation_data=(X_train_seq[vl], y_train.values.ravel()[vl]),
        callbacks=[es], verbose=1,
        use_multiprocessing=True, workers=4
    )
    
    p = (m.predict(X_train_seq[vl])>0.5).astype(int)
    sg_acc.append(accuracy_score(y_train.values.ravel()[vl], p))
print('BiLSTM+SkipGram CV Accuracies:', sg_acc)
print('Mean:', np.mean(sg_acc))

In [ ]:
# Final train & test eval for Skip-Gram
model_sg = build_bilstm(emb_sg)
history_sg = model_sg.fit(
    X_train_seq, y_train.values.ravel(),
    epochs=30, batch_size=128,
    validation_split=0.1, callbacks=[es], verbose=1,
    use_multiprocessing=True, workers=4
)
acc_sg = model_sg.evaluate(X_test_seq, y_test)[1]
print(f"Test Accuracy SkipGram: {acc_sg}")

In [ ]:
# Visualization: CM for Skip-Gram
yp_sg = (model_sg.predict(X_test_seq)>0.5).astype(int)
fig,ax = plt.subplots(1,1,figsize=(5,5))
cm = confusion_matrix(y_test, yp_sg)
ax.imshow(cm, cmap='Oranges')
ax.set_title('BiLSTM+SkipGram')
ax.set_xticks([0,1]); ax.set_xticklabels(['neg','pos'])
ax.set_yticks([0,1]); ax.set_yticklabels(['neg','pos'])
plt.xlabel('Pred')
plt.ylabel('True')

# Add text annotations to the plot
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        text_color = "white" if cm[i, j] > cm.max() / 2 else "black" # Adjust text color for contrast
        ax.text(j, i, format(cm[i, j], 'd'),  # 'd' for integer format
                ha="center", va="center",
                color=text_color)

plt.show()

In [ ]:
# Skip-Gram BiLSTM classification report
y_pred_sg = (model_sg.predict(X_test_seq) > 0.5).astype(int)
print("Classification Report: BiLSTM + Skip-Gram")
print(classification_report(
    y_test,
    y_pred_sg,
    target_names=['negative','positive']
))

In [ ]:
# Plot Skip-Gram metrics
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_sg.history['accuracy'], label='train_acc')
plt.plot(history_sg.history['val_accuracy'], label='val_acc')
plt.title('Skip-Gram BiLSTM Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_sg.history['loss'], label='train_loss')
plt.plot(history_sg.history['val_loss'], label='val_loss')
plt.title('Skip-Gram BiLSTM Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Save the final CBOW-based BiLSTM
model_cbow.save('bilstm_cbow.h5')
print("Saved CBOW BiLSTM to bilstm_cbow.h5")

# Save the final Skip-Gram–based BiLSTM
model_sg.save('bilstm_skipgram.h5')
print("Saved Skip-Gram BiLSTM to bilstm_skipgram.h5")

# Save the Word2Vec models
w2v_cbow.save('w2v_cbow.model')
print("Saved CBOW Word2Vec model to w2v_cbow.model")
w2v_sg.save('w2v_sg.model')
print("Saved Skip-Gram Word2Vec model to w2v_sg.model")

# Save Keras Tokenizer
import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Saved tokenizer to tokenizer.pkl")


# Summary

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Gather predictions
preds = {
    'SVM-CountVec': y_pred_cv,
    'SVM-TFIDF':    y_pred_tv,
    'BiLSTM-CBOW':  y_pred_cbow.ravel(),
    'BiLSTM-SG':    y_pred_sg.ravel()
}

# Compute metrics
rows = []
for name, y_pred in preds.items():
    rows.append({
        'Model':    name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred)
    })

# Build DataFrame
summary_df = pd.DataFrame(rows).set_index('Model')

# Display
print(summary_df)


In [ ]:
results=pd.DataFrame({
    'Model':['LinearSVC-CountVec','LinearSVC-Tfidf','BiLSTM-CBOW','BiLSTM-SkipGram'],
    'Accuracy':[accuracy_score(y_test,y_pred_cv),accuracy_score(y_test,y_pred_tv),acc_cbow,acc_sg]
})
print(results)
plt.figure()
plt.bar(results['Model'],results['Accuracy'])
plt.xticks(rotation=45)
plt.ylabel('Accuracy')
plt.title('Model Comparison')
plt.show()
